# Kaggle setup — verificación de procedencia ISIC 2020

Plantilla de F1.8 (proyecto *melanoma*). Antes de correr:

1. **Add Data → Competitions → `SIIM-ISIC Melanoma Classification`** (distribución oficial, `jpeg/train/`).
2. **Add Data → Your Datasets → `melanoma-isic2020-splits`** (privado: manifiesto + splits, sin imágenes).

Qué hace: toma 200 `image_id` del manifiesto con semilla fija, calcula el SHA256 del JPEG que Kaggle
tiene para cada uno y lo compara con `sha256_original` del manifiesto (calculado en la laptop sobre la
descarga oficial de ISIC). **Si un solo hash no coincide, el notebook falla** y no se debe entrenar nada
hasta resolverlo: el modelo tiene que ver exactamente las imágenes que se auditaron.

Licencia de los datos: CC-BY-NC 4.0. Atribución en `docs/DATA.md` del repositorio.

In [ ]:
# Parámetros (mismos valores que configs/data/isic2020.yaml → data.verify)
COMPETITION_DIR = "/kaggle/input/siim-isic-melanoma-classification/jpeg/train"
SPLITS_DIR = "/kaggle/input/melanoma-isic2020-splits"
SAMPLE_SIZE = 200
SEED = 20260904
EXPECTED = {"images": 33126, "positives": 584, "patients": 2056}

In [ ]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

manifest = pd.read_csv(f"{SPLITS_DIR}/isic2020.csv", dtype={"image_id": str, "patient_id": str})
splits = {
    name: Path(f"{SPLITS_DIR}/{name}.txt").read_text().split() for name in ("train", "val", "test")
}

assert len(manifest) == EXPECTED["images"], len(manifest)
assert manifest["target"].sum() == EXPECTED["positives"]
assert manifest["patient_id"].nunique() == EXPECTED["patients"]
assert sum(map(len, splits.values())) == len(manifest)
assert set().union(*map(set, splits.values())) == set(manifest["image_id"])
print({k: len(v) for k, v in splits.items()})

# Verificar que los archivos de split coinciden con los SHA256 publicados en el dataset
sums_text = Path(f"{SPLITS_DIR}/SHA256SUMS").read_text().splitlines()
sums = dict(line.split()[::-1] for line in sums_text)
for name in splits:
    actual = hashlib.sha256(Path(f"{SPLITS_DIR}/{name}.txt").read_bytes()).hexdigest()
    assert sums[f"{name}.txt"] == actual, f"{name}.txt no coincide con SHA256SUMS"
print("manifiesto y splits íntegros")

In [ ]:
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


rng = np.random.default_rng(SEED)
idx = rng.choice(len(manifest), size=SAMPLE_SIZE, replace=False)
sample = manifest.iloc[idx].sort_values("image_id")

mismatches = []
missing = []
for row in sample.itertuples(index=False):
    path = Path(COMPETITION_DIR) / f"{row.image_id}.jpg"
    if not path.exists():
        missing.append(row.image_id)
        continue
    if sha256_file(path) != row.sha256_original:
        mismatches.append(row.image_id)

print(f"muestra: {len(sample)}  faltantes: {len(missing)}  hashes distintos: {len(mismatches)}")
if missing or mismatches:
    raise RuntimeError(
        "PROCEDENCIA ROTA: la copia de Kaggle no es byte-idéntica a la auditada localmente. "
        f"faltantes={missing[:10]} distintos={mismatches[:10]}. NO ENTRENAR."
    )
print(f"OK: {SAMPLE_SIZE}/{SAMPLE_SIZE} imágenes de Kaggle coinciden byte a byte con el manifiesto")

## Resultado

Anotar en `docs/DATA.md` (sección *Resultado de la verificación local vs. Kaggle*) la fecha y la salida
de la celda anterior. A partir de aquí los notebooks de entrenamiento (F2) parten de esta plantilla y
leen únicamente `train.txt` y `val.txt`; `test.txt` queda reservado a `scripts/evaluate_test.py` (F4).